In [1]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRanker

In [2]:
# ---------------------
# ① データ読み込み
# ---------------------
transactions = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/transactions_train.csv")
customers = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/customers.csv")
articles = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/articles.csv")
sample_submission = pd.read_csv("/Users/kurokawa/Desktop/kaggle/H&M/sample_submission.csv")

In [3]:
# ---------------------
# ① データ読み込み（既に読み込んでいる前提）
# ---------------------
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

# 正例（直近30日）
positive = transactions[transactions['t_dat'] >= '2020-09-15'][['customer_id', 'article_id']].drop_duplicates()
positive['target'] = 1

# 負例作成（1:5の比率）
np.random.seed(42)
neg = pd.DataFrame({
    'customer_id': np.random.choice(positive['customer_id'].unique(), size=len(positive)*5),
    'article_id': np.random.choice(positive['article_id'].unique(), size=len(positive)*5),
})
neg['target'] = 0

# 重複排除（正例と同じ customer_id, article_id のペアを除く）
neg = neg[~neg.set_index(['customer_id', 'article_id']).index.isin(
    positive.set_index(['customer_id', 'article_id']).index
)]

# ---------------------
# ② データ結合と特徴量作成
# ---------------------
# 正負例を結合
data = pd.concat([positive, neg], ignore_index=True)

# 顧客・商品情報を結合
data = data.merge(articles, on='article_id', how='left')
data = data.merge(customers, on='customer_id', how='left')

# カテゴリ変数を category 型に変換
for col in data.select_dtypes(include='object').columns:
    data[col] = data[col].astype('category')

# ---------------------
# ③ 特徴量とターゲット・group作成
# ---------------------
drop_cols = ['customer_id', 'article_id', 'target']
X = data.drop(columns=drop_cols)
y = data['target']

# Group情報：customer_idごとに並べ替え、group数取得
data_sorted = data.sort_values('customer_id').reset_index(drop=True)
X = X.loc[data_sorted.index]
y = y.loc[data_sorted.index]
groups = data_sorted.groupby('customer_id', observed=True).size().tolist()

# ---------------------
# ④ ランカー学習（LambdaRank）
# ---------------------
ranker = LGBMRanker(
    objective='lambdarank',
    metric='ndcg',  # or 'ndcg'
    boosting_type='gbdt',
    random_state=42,
    n_estimators=100,
    max_cat_threshold=128,  # カテゴリが多すぎる場合の警告抑制
    max_bin=255,            # 通常のbin数を明示
)

# 学習（verbose は削除）
ranker.fit(
    X, y,
    group=groups
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011775 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 54894
[LightGBM] [Info] Number of data points in the train set: 1419515, number of used features: 30
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

LGBMRanker(max_bin=255, max_cat_threshold=128, metric='ndcg',
           objective='lambdarank', random_state=42)

In [11]:
X.head()

,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,...,section_name,garment_group_no,garment_group_name,detail_desc,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,794321,SULIMA jkt,262,Jacket,Garment Upper body,1010016,Solid,15,Greyish Beige,4,...,Ladies H&M Sport,1001,Unknown,Outdoor jacket in woven fabric with a double-l...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
1,875272,Boyfriend Shirt,259,Shirt,Garment Upper body,1010023,Denim,7,Grey,2,...,Ladies Denim,1016,Trousers Denim,"Wide shirt in sturdy, organic cotton denim wit...",NaN,NaN,ACTIVE,NONE,50.0,e5b5b43538d04248703523af58a74cd877dd1261a9ed80...
2,875272,Boyfriend Shirt,259,Shirt,Garment Upper body,1010023,Denim,73,Dark Blue,2,...,Ladies Denim,1016,Trousers Denim,"Wide shirt in sturdy, organic cotton denim wit...",NaN,NaN,ACTIVE,NONE,50.0,e5b5b43538d04248703523af58a74cd877dd1261a9ed80...
3,800691,Claudine rib t-shirt,255,T-shirt,Garment Upper body,1010016,Solid,10,White,3,...,Womens Everyday Basics,1002,Jersey Basic,"Fitted T-shirt in stretch, ribbed organic cott...",NaN,NaN,ACTIVE,NONE,29.0,31028c4dea9a87ef366906938b6e1e2fe104eeded193e7...
4,800691,Claudine rib t-shirt,255,T-shirt,Garment Upper body,1010016,Solid,9,Black,4,...,Womens Everyday Basics,1002,Jersey Basic,"Fitted T-shirt in stretch, ribbed organic cott...",NaN,NaN,ACTIVE,NONE,29.0,31028c4dea9a87ef366906938b6e1e2fe104eeded193e7...


In [6]:
# ---------------------
# ⑤ スコア予測
# ---------------------
data['pred_score'] = ranker.predict(X)

# ---------------------
# ⑥ 上位12件推薦（customer_id単位）
# ---------------------
top12 = (
    data.groupby('customer_id')
    .apply(lambda x: x.sort_values('pred_score', ascending=False).head(12))
    .reset_index(drop=True)
)

/var/folders/bf/5s1y_hg57vldk9jqhsj92l8h0000gn/T/ipykernel_12145/2469522748.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data.groupby('customer_id')
/var/folders/bf/5s1y_hg57vldk9jqhsj92l8h0000gn/T/ipykernel_12145/2469522748.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sort_values('pred_score', ascending=False).head(12))


In [7]:
# customer_idごとに推薦されたarticle_idの数をカウント
recommendation_counts = (
    top12.groupby('customer_id')['article_id']
    .count()
    .reset_index(name='num_recommendations')
)

# 12個未満の推薦しかされていないcustomer_idを抽出
under_12 = recommendation_counts[recommendation_counts['num_recommendations'] < 12]

# 件数と一部のデータを表示
print(f"✅ 12個未満の推薦がある顧客数: {len(under_12)}")
display(under_12.head())

✅ 12個未満の推薦がある顧客数: 3475


/var/folders/bf/5s1y_hg57vldk9jqhsj92l8h0000gn/T/ipykernel_12145/1638575377.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  top12.groupby('customer_id')['article_id']


,customer_id,num_recommendations
20,001c1f8d70782f450524d3b3f404474dbd4a7d0d2ad78a...,10
26,0024dbfbe9bf4f2db0ec54e1ce39ecfa91759fd29fa556...,11
49,0038518fc66b22eafe13b0e80c5bf8e13e85c03e434a2c...,10
90,0058caa8e25099d45288038fe4b23efbe030c41277f76a...,11
102,0063a5fab642a52b80dcc5561b3a2ef5a06f13f2967a7a...,11


In [8]:
# ---------------------
# ⑤ 年齢層別の人気商品を作成
# ---------------------
customers = customers[~customers['age'].isna()].copy()
customers['age_group3'] = pd.cut(customers['age'], bins=[0, 30, 60, 200], labels=['~30', '31-60', '60~'])

transactions = transactions.merge(customers[['customer_id', 'age_group3']], on='customer_id', how='left')
recent = transactions[transactions['t_dat'] >= '2020-09-14']

top_items_by_age = (
    recent.groupby('age_group3')['article_id']
    .value_counts()
    .groupby(level=0)
    .head(50)
    .reset_index()
    .rename(columns={'article_id': 'article_id_raw'})
)
top_items_by_age['article_id'] = top_items_by_age['article_id_raw'].astype(str).str.zfill(10)
age_group_popular_dict = top_items_by_age.groupby('age_group3')['article_id'].apply(list).to_dict()

# fallback（全体で人気な商品リスト）
popular_articles_100 = transactions['article_id'].value_counts().head(100).index.astype(str).str.zfill(10).tolist()

# ---------------------
# ⑥ 補完処理の適用
# ---------------------
top12['article_id'] = top12['article_id'].astype(str).str.zfill(10)
recommendations_df = (
    top12.groupby('customer_id')['article_id']
    .apply(list)
    .reset_index()
)

# 年齢層をマージ
age_map = customers[['customer_id', 'age_group3']]
recommendations_df = recommendations_df.merge(age_map, on='customer_id', how='left')

# 補完関数
def complete_with_agegroup(row):
    article_list = row['article_id']
    age_group = row['age_group3']
    if len(article_list) >= 12:
        return article_list[:12]
    fallback = age_group_popular_dict.get(age_group, popular_articles_100)
    fill_items = [a for a in fallback if a not in article_list]
    return article_list + fill_items[:12 - len(article_list)]

# 補完の実行
recommendations_df['prediction'] = recommendations_df.apply(complete_with_agegroup, axis=1)
recommendations_df['prediction'] = recommendations_df['prediction'].apply(lambda x: ' '.join(x))

print("✅ 年齢層ごとの補完を反映済みの推薦リストが完成しました")
recommendations_df.head()

/var/folders/bf/5s1y_hg57vldk9jqhsj92l8h0000gn/T/ipykernel_12145/1846273080.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  recent.groupby('age_group3')['article_id']
/var/folders/bf/5s1y_hg57vldk9jqhsj92l8h0000gn/T/ipykernel_12145/1846273080.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(level=0)
/var/folders/bf/5s1y_hg57vldk9jqhsj92l8h0000gn/T/ipykernel_12145/1846273080.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and 

✅ 年齢層ごとの補完を反映済みの推薦リストが完成しました


,customer_id,article_id,age_group3,prediction
0,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,"[0936895002, 0881112001, 0864372001, 084579000...",~30,0936895002 0881112001 0864372001 0845790005 08...
1,00039306476aaf41a07fed942884f16b30abfa83a2a8be...,"[0917843001, 0875978001, 0852678003, 091429300...",~30,0917843001 0875978001 0852678003 0914293003 08...
2,0003e867a930d0d6842f923d6ba7c9b77aba33fe2a0fbf...,"[0852584002, 0862316001, 0871992002, 089262400...",31-60,0852584002 0862316001 0871992002 0892624001 08...
3,00040239317e877c77ac6e79df42eb2633ad38fcac09fc...,"[0935752001, 0883454002, 0833408001, 085264300...",31-60,0935752001 0883454002 0833408001 0852643006 08...
4,000493dd9fc463df1acc2081450c9e75ef8e87d5dd17ed...,"[0898555001, 0905803004, 0911197001, 090437400...",~30,0898555001 0905803004 0911197001 0904374002 08...


In [9]:
# prediction をリストに変換（空白区切り）
recommendations_df['pred_list'] = recommendations_df['prediction'].apply(lambda x: x.split())

# 12個未満のレコードだけを抽出
under_12 = recommendations_df[recommendations_df['pred_list'].apply(len) < 12]

# 結果を表示
print(f"✅ 12個未満の商品を推薦されている顧客数: {len(under_12)}")
display(under_12.head())


✅ 12個未満の商品を推薦されている顧客数: 0


,customer_id,article_id,age_group3,prediction,pred_list


In [10]:
# ---------------------
# ⑧ 提出ファイル作成（補完あり）
# ---------------------
# sample_submission に推薦結果をマージ
submission = sample_submission[['customer_id']].merge(recommendations_df[['customer_id', 'prediction']], on='customer_id', how='left')

# 人気商品で欠損を補完
fallback_12 = ' '.join(popular_articles_100[:12])  # or popular_articles[:12]
submission['prediction'] = submission['prediction'].fillna(fallback_12)

# CSV 出力
submission.to_csv("lgbm_articles_Labmdarank_ndcg_5th.csv", index=False)
print("✅ lgbm_articles_Labmdarank_ndcg_5th.csv を保存しました")


✅ lgbm_articles_Labmdarank_ndcg_5th.csv を保存しました


In [ ]:
submission.head()